# How LLMs Use Scratchpad Notes for Online Learning with Expert Advice

We study how DeepSeek V3 uses an explicit scratchpad ("note") to retain state across rounds in a two-turn online learning protocol with expert advice. In each round, four experts make binary predictions; the model sees their predictions, makes its own, then receives feedback and updates a free-form note for future use. This note is the model's only memory between rounds.

We analyze note behavior across three datasets:

| Dataset | Best expert | Other experts | Key challenge |
|---------|------------|---------------|---------------|
| **Stratified** | 90–100% | 45–80% (tiered) | Identify and lock onto dominant expert |
| **Flat** | 60–70% | 40–60% | Distinguish experts with small accuracy gaps |
| **Anti-signal** | 60–70% | 0–10% (anti) + 40–60% (mid) | Detect and avoid adversarial expert |

All experiments: DeepSeek V3, v2 prompts with hint ("Use past outcomes to figure out which experts are most trustworthy"), 30 cases, 100 steps each, two prompt framings (weather and online).

## 1. Note Format Evolution

We classify each note into five categories:

- **Cumulative counter**: `A:5/25, B:20/25` — running accuracy tallies (most useful)
- **Conditional counter**: `A:sunny(22/39), B:rainy(28/55)` — accuracy split by weather outcome (**weather framing only**; causes information loss)
- **Last-step-only**: `A correct, B wrong` — only the most recent round (no memory)
- **Round listing**: `D correct R1,R2,R6,R8; wrong R9,R11` — enumerates rounds by index (wastes tokens, eventually truncated)
- **Qualitative**: `B often correct, A rarely correct` — verbal summary without counts

Conditional counter is **exclusive to weather framing** — the sunny/rainy labels tempt the model into splitting accuracy by weather outcome. Online framing uses abstract 0/1 labels and never exhibits this failure mode.

### Table 1a: Note format evolution — Weather framing (% of 30 cases)

| Step | Strat: counter | Strat: cond. | Strat: last-step | Flat: counter | Flat: cond. | Flat: last-step | Anti: counter | Anti: cond. | Anti: last-step |
|------|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 1  | 37% | 27% | 23% | 20% | 10% | 57% | 20% | 17% | 53% |
| 10 | 43% | 30% | 13% | 47% | 13% | 30% | 30% | 17% | 43% |
| 25 | 43% | 30% | 13% | 60% | 20% | 13% | 47% | 17% | 27% |
| 50 | 47% | 30% | 13% | 67% | 23% | 3%  | 47% | 23% | 23% |
| **99** | **53%** | **33%** | **7%** | **70%** | **23%** | **0%** | **50%** | **23%** | **20%** |

### Table 1b: Note format evolution — Online framing (% of 30 cases)

| Step | Strat: counter | Strat: last-step | Flat: counter | Flat: last-step | Anti: counter | Anti: last-step | Anti: qualitative |
|------|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 1  | 0%  | 70% | 0%  | 80% | 0%  | 77% | 0%  |
| 10 | 30% | 40% | 43% | 33% | 30% | 40% | 7%  |
| 25 | 53% | 33% | 70% | 20% | 53% | 13% | 10% |
| 50 | 60% | 27% | 77% | 13% | 73% | 3%  | 10% |
| **99** | **70%** | **23%** | **90%** | **7%** | **77%** | **3%** | **10%** |

Minor categories (round listing, other) omitted for clarity.

**Findings**:
- All conditions start with mostly last-step-only notes and transition to cumulative counters between steps 10–25.
- Online framing achieves higher counter adoption at step 99 (70–90%) than weather (50–70%), because weather loses 23–33% of cases to the conditional counter failure mode.
- Anti-signal has the highest last-step-only rate in weather at step 99 (20%) — confusing expert signals disrupt note organization.
- Flat-online achieves the highest counter rate (90%).

### Table 2: When cumulative counters first appear

| Metric | Strat-W | Strat-O | Flat-W | Flat-O | Anti-W | Anti-O |
|--------|:-:|:-:|:-:|:-:|:-:|:-:|
| Median step | 0 | 12 | 7 | 11 | 3 | 19 |
| Mean step | 8 | 21 | 12 | 20 | 14 | 20 |
| Adoption rate | 87% | 70% | 93% | 90% | 73% | 80% |

- Weather framing adopts counters earlier (median step 0–7) because the first feedback already uses fraction-like language (e.g., `A:rainy(1/2)`).
- Online framing starts with pure text ("A wrong, B correct") and takes 10–20 steps to transition.
- Anti-signal weather has the lowest adoption rate (73%).

## 2. The Conditional Counter Problem

In weather framing, 23–33% of cases develop a "conditional counter" that splits accuracy by weather outcome instead of tracking overall accuracy.

**Example** (Anti-signal, Case 0, Step 99):
```
A:sunny(22/39); B:rainy(28/55); C:sunny(28/56); D:rainy(6/60)
```
Each expert's accuracy is only recorded for one weather condition. Denominators (39, 55, 56, 60) don't sum to 100; the other condition's data is lost entirely.

**Correct format**: `A:42/100, B:70/100, C:60/100, D:9/100`

### Table 3: Final denominator accuracy at step 99 (% of 30 cases)

| Range | Strat-W | Strat-O | Flat-W | Flat-O | Anti-W | Anti-O |
|-------|:-:|:-:|:-:|:-:|:-:|:-:|
| 90–100 (correct) | **47%** | **37%** | **43%** | **47%** | 27% | **37%** |
| 50–89 (partial) | 20% | 23% | 37% | 27% | 27% | 30% |
| < 50 (degenerated) | 23% | 10% | 13% | 20% | 20% | 13% |
| No denominator | 10% | 27% | 7% | 7% | 27% | 17% |

Only 27–47% of cases maintain correct denominators at step 99. Anti-signal weather is worst (27%).

## 3. Note Quality and Regret

### Table 4: Mean regret by final note format

| Dataset | Weather: counter | Weather: non-counter | Online: counter | Online: non-counter |
|---------|:-:|:-:|:-:|:-:|
| Stratified | **7.8** (n=26) | 10.0 (n=4) | 9.9 (n=21) | 7.9 (n=9) |
| Flat | **7.1** (n=28) | 12.0 (n=2) | **7.8** (n=27) | 9.3 (n=3) |
| Anti-signal | **12.6** (n=22) | 15.1 (n=8) | **11.7** (n=25) | 9.6 (n=5) |

Counter-based notes are associated with lower regret in 5 of 6 conditions. The exception (stratified online) has a small non-counter sample (n=9) that includes cases where a simple "B correct, others wrong" note happened to work because the best expert was 90%+ accurate.

### Table 5: Note vs no-note regret gap across datasets

| Dataset | Best expert | Note regret (avg) | No-note regret (avg) | Gap |
|---------|:-:|:-:|:-:|:-:|
| Stratified | 90–100% | 8.7 | 13.3 | **4.6** |
| Flat | 60–70% | 7.7 | 8.4 | **0.7** |
| Anti-signal | 60–70% | 12.3 | 19.7 | **7.4** |

Note value is **not** simply a function of best-expert accuracy. Anti-signal and flat share the same best-expert range (60–70%), yet the note gap differs by 10x (7.4 vs. 0.7). Notes are most valuable when there is an expert that must be **actively identified and avoided**.

## 4. Does the Model Discover and Exploit the Anti-Signal Expert?

The anti-signal expert (0–10% accuracy) is wrong 90–100% of the time. Inverting its predictions would yield higher accuracy than even the best expert. We scan all 18,000 notes across all three datasets for evidence of this insight.

### Table 6: Keyword scan — strategic vs descriptive language

| Keyword category | Stratified (6000 notes) | Flat (6000 notes) | Anti-signal (6000 notes) |
|-----------------|:-:|:-:|:-:|
| **Strategic (invert/revert/opposite/flip/negate)** | **0** | **0** | **0** |
| "rarely correct" | 0 | 0 | 96 (1 case) |
| "unreliable" | 1 (1 case) | 6 (1 case) | 61 (3 cases) |
| Notes with 0/N accuracy (N≥50) | 0 | 0 | 89 |

**Across all 18,000 notes, zero contain strategic language about inverting or exploiting any expert.**

The model's awareness of poor-performing experts takes three forms, all purely descriptive:

1. **Near-zero counters** (weather, anti-signal): `B:1/50`, `D:0/85` — recorded but never interpreted strategically
2. **"Rarely correct"** (online, anti-signal Case 5): `A rarely correct` — a qualitative label appended to counters; model follows the best expert, not the inverse of A
3. **"Unreliable"** (online, anti-signal Cases 8, 19, 26): `B reliable; C,D unreliable` — leads to following B, not inverting C/D

### Incidental Inversion

We define an "inversion step" as: the anti-signal expert and ≥2 others predict X (majority = X), but the model predicts the opposite.

### Table 7: Inversion frequency on anti-signal dataset

| Metric | Weather | Online |
|--------|:-:|:-:|
| Cases with any inversion | 30/30 (100%) | 29/30 (97%) |
| Total inversion steps | 210/3000 (7%) | 278/3000 (9%) |

Inversion occurs in nearly every case but at a low rate. All observed inversions are **side effects of following the best expert**, who happens to disagree with the majority — not deliberate exploitation of the anti-signal expert.

> *Example — Case 3, Step 34*: experts=[1,1,0,1], anti(A)=1, majority=1, model=0  
> Note: `A 5/35, B 16/35, C 22/35, D 16/35`  
> Model follows C (highest count, predicting 0). This coincidentally opposes A, but the reasoning is "follow C" not "invert A".

**Conclusion**: The model's strategy is exclusively **exclusion** (ignore the worst) rather than **exploitation** (invert the worst). Despite recording `0/85` accuracy — clear evidence of a near-perfect negative signal — the model never reasons about using this inversely. This represents a fundamental limitation: LLMs can learn "who to trust" but not "who to distrust and invert".

## 5. Full Baseline Comparison

### Table 8: Final regret at step 100 (30 cases)

| Strategy | Stratified | Flat | Anti-signal |
|----------|:-:|:-:|:-:|
| MW Optimal | 1.6 | 3.3 | 5.9 |
| Follow the Leader | 1.3 | 3.5 | 2.7 |
| **LLM +note (best)** | **8.1** | **7.4** | **11.3** |
| **LLM −note (best)** | **12.7** | **8.2** | **17.5** |
| Follow Previous Winners | 12.5 | 8.6 | 11.7 |
| Majority Vote | 14.5 | 8.4 | 25.2 |
| Random Guessing | 44.5 | 15.2 | 15.1 |

Notable:
- On anti-signal, **Majority Vote (25.2) is worse than Random (15.1)** — the anti-signal expert corrupts uniform voting.
- **LLM without notes (17.5) is also worse than Random** on anti-signal — without memory, the model cannot learn to avoid the adversarial expert.
- **FTL beats MW on anti-signal** (2.7 vs. 5.9) because FTL commits to the single best expert and ignores all others, while MW's weighted voting is polluted by the anti-signal expert's residual weight.

## 6. Summary

1. **Note format converges to cumulative counters** in 50–90% of cases by step 99. Online framing converges more reliably than weather framing, which suffers from a conditional counter failure mode (23–33%) exclusive to weather's sunny/rainy labels.

2. **Denominator accuracy degrades over time**. Only 27–47% of cases maintain correct denominators at step 99.

3. **Counter-based notes predict lower regret** in most conditions.

4. **Note value depends on task structure, not just expert accuracy**. Anti-signal (best 60–70%) benefits from notes more than stratified (best 90–100%), because the note's primary value is identifying and excluding the adversarial expert.

5. **LLMs never discover inversion**. Despite recording 0/85 accuracy, the model never reasons about inverting. Its strategy is limited to exclusion — a gap between pattern recognition ("this expert is always wrong") and strategic reasoning ("therefore predict the opposite").

---

## Appendix: Anti-Signal Expert Identification Accuracy

For anti-signal cases where the final note contains parseable per-expert scores, we check whether the model correctly identifies the anti-signal expert as the worst performer.

### Table A1: Identification accuracy

| Metric | Weather (n=19) | Online (n=23) |
|--------|:-:|:-:|
| Anti-signal identified as worst | **89%** (17/19) | **78%** (18/23) |
| Best expert correctly identified | 37% (7/19) | 65% (15/23) |

### Table A2: Regret by identification accuracy

| Condition | Weather regret | Online regret |
|-----------|:-:|:-:|
| Anti-signal correctly identified | 10.7 | 11.1 |
| Anti-signal NOT identified | 15.5 | 13.0 |
| No parseable scores | 16.9 | 10.9 |

The model reliably identifies the worst expert but is less accurate at identifying the best — particularly in weather framing, where conditional counters distort comparisons.